# PopFrame: расселение и транспортная доступность
## Практикум для слушателей ДПО по транспортному планированию

**Цель:** исследовать систему расселения Ленинградской области, построить агломерации
и зоны опорных пунктов, сравнить результаты при изменении населения и времени доступности.

Для основной части достаточно файла **`Ленинградская область.pickle`**.
Он содержит готовую модель: границу региона, населённые пункты и матрицу времени между ними.

**Порядок работы:** запускайте ячейки сверху вниз кнопкой ▶ или сочетанием Shift+Enter.
Дождитесь окончания текущей ячейки. Меняйте значения в ячейках «Параметры».
После изменения параметров заново выполните следующие ячейки. При перезапуске
среды Colab повторите установку и загрузку данных.

**Маршрут практикума**

1. Подготовка среды и знакомство с моделью.
2. Схема расселения и агломерации.
3. Сценарии изменения населения и времени доступности.
4. Выбор опорных пунктов и построение их зон.
5. Оценка расположения территории и профилей развития.
6. Землепользование, пространственное неравенство и инженерная инфраструктура.
7. Сохранение результатов и самостоятельное задание.

PopFrame помогает исследовать пространственные связи между населёнными пунктами,
сравнивать варианты развития и представлять результаты в таблицах и на картах.

## 1. Подготовка среды
### Установка библиотеки
Устанавливаем PopFrame из ветки разработки **`develop`** с проверенными версиями
GeoPandas 1.0.1 и pandas 2.2.3. Установка может занять
несколько минут. Если Colab предложит перезапустить среду после установки, выполните
перезапуск и продолжите с ячейки импорта.

Ветка обновляется со временем. Версии пакетов и идентификатор коммита автоматически сохраняются
в файл параметров вместе с результатами в разделе 8.

In [ ]:
%pip install -q "geopandas==1.0.1" "pandas==2.2.3" "git+https://github.com/Mvin8/PopFrame.git@develop"

### Инструменты для загрузки
`Path` помогает указать расположение файла, а `Region` загружает модель региона.
Остальные инструменты подключим по мере выполнения соответствующих разделов.

In [ ]:
from pathlib import Path
from popframe.models.region import Region

### Загрузка учебного файла
В Colab следующая ячейка откроет выбор файла при первой загрузке модели.
Выберите **`Ленинградская область.pickle`** из учебного набора.
При повторном запуске ячейки используется уже загруженный файл.
Локально положите файл в папку `data` рядом с ноутбуком.

In [ ]:
DATA_DIR = Path('/content/data') if Path('/content').exists() else Path.cwd() / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_NAME = 'Ленинградская область.pickle'
MODEL_PATH = DATA_DIR / MODEL_NAME

if not MODEL_PATH.exists() and Path('/content').exists():
    from google.colab import files
    uploaded = files.upload()
    if MODEL_NAME in uploaded:
        MODEL_PATH.write_bytes(uploaded[MODEL_NAME])
    del uploaded

if not MODEL_PATH.exists():
    raise FileNotFoundError(f'Поместите файл {MODEL_NAME} в {DATA_DIR} и повторите ячейку.')
region_source = Region.from_pickle(str(MODEL_PATH))
print('Модель загружена:', MODEL_PATH.name)

### Подготовка модели к анализу
`prepare_for_analysis()` проверяет геометрии, население и соответствие ID в матрице
и таблице пунктов. Метод создаёт рабочую копию в локальной проекции UTM (метры)
и определяет размерные категории пунктов по населению.

Параметр `repair_geometry=True` включает исправление самопересечений границы
с сохранением её площадных частей. Исходная модель остаётся доступной отдельно.

Матрица хранит **время в минутах**. Ноль на диагонали соответствует поездке пункта
в самого себя, бесконечность обозначает недостижимую пару.
После подготовки модель готова к построению карт и расчёту сценариев.

In [ ]:
region_model = region_source.prepare_for_analysis(repair_geometry=True)
towns = region_model.get_towns_gdf()
boundary = region_model.region
travel_times = region_model.accessibility_matrix
LOCAL_CRS = region_model.crs

print('Модель готова к анализу. Проекция:', LOCAL_CRS)

### Знакомство с данными
Посмотрим крупнейшие населённые пункты и небольшой фрагмент матрицы. Строка — пункт отправления, столбец — пункт назначения; значения — минуты.

In [ ]:
print(f'Населённых пунктов: {len(towns):,}')
print(f'Население в наборе: {towns.population.sum():,} чел.')
display(towns[['id', 'name', 'population', 'level']].nlargest(15, 'population'))
display(travel_times.iloc[:5, :5])

### Параметры основной практики

- `TOWN_ID`: ID исследуемого пункта из таблицы выше; `None` выбирает крупнейший.
- `BASE_TIME`: базовый порог агломераций в минутах, от 50 и выше.
- `SCENARIO_TIME`: альтернативный порог для сравнения зон доступности по одной матрице.
- `SCENARIO_POPULATION`: население исследуемого пункта в учебном сценарии;
  `None` задаёт увеличение на 50%.
- `ANCHOR_IDS`: список ID опорных пунктов, например `[12, 34]`.
  Пустой список выбирает два крупнейших пункта внутри региона.
- `ANCHOR_TIME`: порог времени для зон опорных пунктов, минуты.

**Задание:** выполните ноутбук с исходными параметрами, затем измените один параметр.
Сформулируйте гипотезу и сравните результаты расчётов.

In [ ]:
TOWN_ID = None
BASE_TIME = 80
SCENARIO_TIME = 100
SCENARIO_POPULATION = None
ANCHOR_IDS = []
ANCHOR_TIME = 50

selected_town_id = int(towns.population.idxmax()) if TOWN_ID is None else TOWN_ID
assert selected_town_id in towns.index, 'Выберите ID из таблицы населённых пунктов.'
assert BASE_TIME >= 50 and SCENARIO_TIME >= 50, 'Порог агломераций должен быть не меньше 50 минут.'
assert ANCHOR_TIME > 0, 'Порог опорных пунктов должен быть положительным.'
selected_town = towns.loc[selected_town_id]
scenario_population = (int(round(selected_town.population * 1.5))
                       if SCENARIO_POPULATION is None else SCENARIO_POPULATION)
assert isinstance(scenario_population, int) and scenario_population > 0, 'Население — положительное целое число.'
print(f'Исследуемый пункт: {selected_town["name"]} (ID {selected_town_id})')
print(f'Учебный сценарий населения: {selected_town.population:,} → {scenario_population:,} чел.')

### Чтение матрицы доступности
Строка исследуемого пункта показывает время **из него** в остальные пункты.
Столбец показывает время поездки в обратном направлении.
Матрица помогает найти доступные пункты и сравнить их транспортное положение.

**Вопрос:** какие пункты доступны за базовое время? Совпадают ли ближайшие по времени
пункты с ближайшими на карте?

In [ ]:
times_from_town = travel_times.loc[selected_town_id]
reachable = towns.loc[times_from_town.le(BASE_TIME)].copy()
reachable['Время, мин'] = times_from_town.reindex(reachable.index)
display(reachable[['name', 'population', 'Время, мин']].sort_values('Время, мин').head(15))
print(f'Доступно за {BASE_TIME} мин: {len(reachable)} пунктов, включая исходный.')

## 2. Схема расселения
### `PopulationFrame.build_circle_frame()`
**Вход:** модель региона с точками, населением и размерными категориями.
**Принцип:** вокруг каждой точки строится условный круг, радиус которого зависит
от корня из численности населения и категории.
**Результат:** слой полигонов, наглядно представляющий структуру расселения.

Размеры кругов помогают сопоставить населённые пункты и выделить крупные центры.
Для определения принадлежности пунктов к агломерациям используются их исходные точки.

**Задание:** найдите крупнейшие центры и участки с малым количеством населённых пунктов.

In [ ]:
from popframe.method.popuation_frame import PopulationFrame
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.figsize': (11, 7), 'font.size': 11})

population_frame = PopulationFrame(region=region_model).build_circle_frame()
fig, ax = plt.subplots()
boundary.boundary.plot(ax=ax, color='#555555', linewidth=0.7)
population_frame.plot(ax=ax, color='#3976a8', alpha=0.55, edgecolor='white', linewidth=0.2)
ax.set_title('Схема расселения: условные круги по численности населения')
ax.set_axis_off()
plt.tight_layout()
plt.show()

## 3. Агломерации
### `AgglomerationBuilder.get_agglomerations()`
**Вход:** населённые пункты, матрица времени и параметр `time` в минутах.
Метод выбирает центры по размерной категории и населению, находит доступные пункты,
строит вокруг них буферы по оставшемуся времени, объединяет пересечения
и формирует границы агломераций в пределах региона.

Исходный порог `time` уменьшается на 10 минут при переходе к каждой следующей
размерной категории центров. Для перевода оставшегося времени в радиус буфера
используется коэффициент **400 м/мин**. Минимальное население центра — **15 000 человек**.

**Результат:** геометрия, центры (`core_cities`), население (`population`) и уровень
агломерации (`agglomeration_level`, 1–5), определяемый численностью населения.
Полученные слои позволяют сравнить масштаб агломераций, их центры и охват населённых пунктов.

In [ ]:
from popframe.method.agglomeration import AgglomerationBuilder

base_builder = AgglomerationBuilder(region=region_model)
base_agglomerations = base_builder.get_agglomerations(time=BASE_TIME, raise_on_empty=False)
base_status = base_builder.evaluate_city_agglomeration_status(towns, base_agglomerations)
display(base_agglomerations.drop(columns='geometry').head(15))
display(base_status[['name', 'population', 'agglomeration_status', 'agglomeration_level']].head(15))

### Карта агломераций и принадлежность пунктов
`evaluate_city_agglomeration_status()` определяет, пересекает ли **точка** пункта
полигон агломерации, и добавляет статус: центр, участник или вне агломерации.
Таблицу статусов можно использовать для сравнения сценариев по одним и тем же ID.

Следующая служебная ячейка задаёт одинаковые цвета и легенду для всех сценариев.
Редактировать её для выполнения заданий не требуется.

In [ ]:
from matplotlib.patches import Patch

AGGLO_COLORS = {1: '#eff3ff', 2: '#bdd7e7', 3: '#6baed6', 4: '#3182bd', 5: '#08519c'}

def plot_agglomerations(areas, town_points, title, ax):
    boundary.boundary.plot(ax=ax, color='#777777', linewidth=0.6)
    for level, color in AGGLO_COLORS.items():
        subset = areas.loc[areas['agglomeration_level'].eq(level)]
        if not subset.empty:
            subset.plot(ax=ax, color=color, edgecolor='#555555', linewidth=0.5)
    town_points.plot(ax=ax, color='#333333', markersize=3, alpha=0.6)
    ax.set_title(title)
    ax.set_axis_off()

fig, ax = plt.subplots()
plot_agglomerations(base_agglomerations, towns, f'Агломерации: базовый порог {BASE_TIME} мин', ax)
ax.legend(handles=[Patch(facecolor=c, label=f'Уровень {level}') for level, c in AGGLO_COLORS.items()],
          loc='upper left', title='Уровень по населению')
plt.tight_layout()
plt.show()
if base_agglomerations.empty:
    print('При этих параметрах агломерации не получены. Это допустимый результат.')

**Вопросы к результату:** какие центры образовали общую агломерацию?
Как распределены населённые пункты внутри и вне её границ?
Какие направления связи между центрами вы бы исследовали в первую очередь?

## 4. Сценарии: меняем один фактор
### Сценарий А — изменение населения
Передаём `update_df`: таблицу новых значений населения с ID пунктов в индексе.
Библиотека пересчитывает размерную категорию и строит агломерации с обновлёнными данными.
Так можно проследить, как рост выбранного пункта влияет на систему расселения
при сохранении исходной матрицы поездок.

Для каждого сценария создаём отдельный `AgglomerationBuilder`.
Базовые результаты сохраняем для последующего сравнения.

In [ ]:
import pandas as pd

population_update = pd.DataFrame({'population': [scenario_population]}, index=[selected_town_id])
scenario_towns = region_model.get_update_towns_gdf(population_update)
population_builder = AgglomerationBuilder(region=region_model)
population_agglomerations = population_builder.get_agglomerations(
    update_df=population_update, time=BASE_TIME, raise_on_empty=False)
population_status = population_builder.evaluate_city_agglomeration_status(scenario_towns, population_agglomerations)
population_frame_scenario = PopulationFrame(region=region_model).build_circle_frame(population_update)

display(pd.DataFrame({
    'База': towns.loc[selected_town_id, ['name', 'population', 'level']],
    'Сценарий А': scenario_towns.loc[selected_town_id, ['name', 'population', 'level']],
}))

### Сценарий Б — другой порог времени
Используем исходное население и меняем параметр `time`.
Сравнение показывает, как выбранный порог влияет на границы агломераций
и количество охваченных населённых пунктов.

In [ ]:
time_builder = AgglomerationBuilder(region=region_model)
time_agglomerations = time_builder.get_agglomerations(time=SCENARIO_TIME, raise_on_empty=False)
time_status = time_builder.evaluate_city_agglomeration_status(towns, time_agglomerations)

### Сравнение результатов
Сопоставим количество полигонов, число пунктов внутри агломераций
и суммарное население охваченных пунктов. Население суммируется по уникальным ID.

Все карты используют одну шкалу уровней. Таблицы показывают изменения относительно
базы и помогают связать результат с изменением выбранного параметра.

In [ ]:
def summarize_scenario(areas, status):
    inside = status['agglomeration_status'].ne('Вне агломерации')
    return {'Полигонов': len(areas), 'Пунктов внутри': int(inside.sum()),
            'Население пунктов внутри, чел.': int(status.loc[inside, 'population'].sum())}

scenario_summary = pd.DataFrame({
    'База': summarize_scenario(base_agglomerations, base_status),
    'А: население': summarize_scenario(population_agglomerations, population_status),
    'Б: время': summarize_scenario(time_agglomerations, time_status),
}).T
display(scenario_summary)
display(scenario_summary.subtract(scenario_summary.loc['База']).rename_axis('Изменение относительно базы'))

status_comparison = towns[['name', 'population']].copy()
status_comparison['База'] = base_status['agglomeration_status']
status_comparison['А: население'] = population_status['agglomeration_status']
status_comparison['Б: время'] = time_status['agglomeration_status']
changed = status_comparison['База'].ne(status_comparison['А: население']) | status_comparison['База'].ne(status_comparison['Б: время'])
display(status_comparison.loc[changed].head(20))
print('Пунктов, сменивших статус хотя бы в одном сценарии:', int(changed.sum()))

fig, axes = plt.subplots(1, 3, figsize=(18, 7))
for ax, areas, points, title in zip(axes,
        [base_agglomerations, population_agglomerations, time_agglomerations],
        [towns, scenario_towns, towns],
        [f'База: {BASE_TIME} мин', f'А: население, {BASE_TIME} мин', f'Б: время, {SCENARIO_TIME} мин']):
    plot_agglomerations(areas, points, title, ax)
fig.legend(handles=[Patch(facecolor=c, label=f'Уровень {level}') for level, c in AGGLO_COLORS.items()],
           loc='lower center', ncol=5)
fig.tight_layout(rect=(0, 0.08, 1, 1))
plt.show()

**Задание:** опишите изменения границ, уровней и состава агломераций.
Свяжите результат с категорией выбранного пункта, его доступностью и положением
относительно других центров. Отметьте территории, сохранившие свои характеристики.

## 5. Опорные населённые пункты
### `AnchorSettlementBuilder.get_anchor_settlement_boundaries()`
**Вход:** модель региона, слой точек с признаком `is_anchor_settlement`
и порог `time`. Признак позволяет задать исследуемый набор опорных пунктов.

Метод использует матрицу времени, строит и объединяет зоны доступности,
затем разделяет общие области, в том числе с помощью диаграммы Вороного.
Результат помогает сравнить пространственный охват разных наборов опорных пунктов.

В учебном сценарии выбираются два крупнейших пункта внутри региона.
Для исследования своего варианта задайте `ANCHOR_IDS` в ячейке параметров
и повторите расчёт.

In [ ]:
from popframe.method.anchor_settlement import AnchorSettlementBuilder

anchor_towns = towns.copy()
inside_region = towns.geometry.within(boundary.geometry.union_all())
anchor_candidates = towns.loc[inside_region]
anchor_ids = ANCHOR_IDS or anchor_candidates.nlargest(2, 'population').index.tolist()
assert anchor_ids and set(anchor_ids).issubset(anchor_candidates.index), 'Выберите опорные пункты внутри региона.'
anchor_towns['is_anchor_settlement'] = anchor_towns.index.isin(anchor_ids)
display(anchor_towns.loc[anchor_ids, ['id', 'name', 'population']])

anchor_boundaries = AnchorSettlementBuilder(region=region_model).get_anchor_settlement_boundaries(
    towns=anchor_towns, time=ANCHOR_TIME)
assert not anchor_boundaries.empty, 'Зоны не получены: проверьте опорные пункты и матрицу времени.'
assert anchor_boundaries.geometry.is_valid.all(), 'Метод вернул некорректные геометрии.'

fig, ax = plt.subplots()
boundary.boundary.plot(ax=ax, color='#777777', linewidth=0.6)
anchor_boundaries.plot(ax=ax, color='#bdd7e7', edgecolor='#3976a8', alpha=0.6)
anchor_towns.loc[anchor_ids].plot(ax=ax, color='#aa5522', marker='*', markersize=100)
for row in anchor_towns.loc[anchor_ids].itertuples():
    ax.annotate(row.name, (row.geometry.x, row.geometry.y), xytext=(5, 5), textcoords='offset points')
ax.set_title(f'Учебные зоны опорных пунктов: {ANCHOR_TIME} мин')
ax.set_axis_off()
plt.tight_layout()
plt.show()

**Вопросы:** как соотносятся зоны опорных пунктов и агломерации?
Как изменяется охват территории при выборе другого опорного пункта?

## 6. Расположение выбранной территории
### Подготовка полигона
В этом разделе исследуем выбранную площадку. Для работы с гексагонами положите
`LO_hex.geojson` в папку `data` (в Colab — через панель «Файлы»).
При работе только с pickle ноутбук создаст учебный полигон вокруг выбранного пункта.
Его размер задаётся параметром `SITE_RADIUS_M`.

In [ ]:
import geopandas as gpd

HEX_ID = None  # Если есть LO_hex.geojson, укажите ID из его столбца id.
SITE_RADIUS_M = 1000  # Радиус условной площадки, если гексагоны не выданы.

hex_path = DATA_DIR / 'LO_hex.geojson'
if hex_path.exists():
    hexagons = gpd.read_file(hex_path)
    assert hexagons.crs is not None and 'id' in hexagons, 'Нужны CRS и столбец id.'
    assert not hexagons.empty and hexagons['id'].is_unique, 'Нужны гексагоны с уникальными ID.'
    selected_hex_id = hexagons.iloc[0]['id'] if HEX_ID is None else HEX_ID
    territory = hexagons.loc[hexagons['id'].eq(selected_hex_id)].to_crs(LOCAL_CRS).copy()
    assert len(territory) == 1, 'Указанный ID гексагона не найден.'
    territory['name'] = f'Гексагон {selected_hex_id}'
else:
    assert SITE_RADIUS_M > 0
    territory = gpd.GeoDataFrame({'name': [f'Учебная площадка: {selected_town["name"]}']},
        geometry=[towns.loc[selected_town_id].geometry.buffer(SITE_RADIUS_M)], crs=LOCAL_CRS)
    print('Гексагоны не загружены: используется условная площадка.')
assert territory.geometry.geom_type.isin(['Polygon', 'MultiPolygon']).all()
assert territory.geometry.is_valid.all() and not territory.geometry.is_empty.any()
territory = territory.reset_index(drop=True)
display(territory[['name']])

### `TerritoryEvaluation.evaluate_territory_location()`
Метод оценивает положение территории в системе расселения.
Сначала он находит населённые пункты в ближнем буфере и выбирает наибольший балл
их размерной категории, затем проверяет расположение территории между центрами.

**Результат:** балл, текстовое пояснение и сведения о связанных населённых пунктах.
Оценка помогает сопоставлять площадки по их пространственному положению.

**Задание:** сопоставьте пояснение с картой. Какие населённые пункты определяют
положение выбранной территории?

In [ ]:
from popframe.method.territory_evaluation import TerritoryEvaluation

evaluation = TerritoryEvaluation(region=region_model)
location_results = pd.DataFrame(evaluation.evaluate_territory_location(territory))
assert len(location_results) == len(territory)
territory_scored = territory.copy()
territory_scored['score'] = location_results['score'].to_numpy()
display(location_results)

fig, ax = plt.subplots(figsize=(9, 7))
territory.plot(ax=ax, color='#bdd7e7', edgecolor='#3976a8', alpha=0.6)
nearby_towns = towns.loc[towns.intersects(territory.geometry.iloc[0].buffer(15000))]
if not nearby_towns.empty:
    nearby_towns.plot(ax=ax, color='#333333', markersize=20)
    for row in nearby_towns.nlargest(5, 'population').itertuples():
        ax.annotate(row.name, (row.geometry.x, row.geometry.y), xytext=(4, 4), textcoords='offset points')
ax.set_title('Учебная территория и населённые пункты в её окружении')
ax.set_axis_off()
plt.tight_layout()
plt.show()

### `TerritoryEvaluation.calculate_potential()` — профили развития
Метод сравнивает баллы по пяти критериям с требованиями профилей развития:
жилая застройка, промышленность, рекреация и другие.

Для каждого профиля рассчитываются число выполненных критериев и взвешенное
превышение требований. По этим показателям формируется ранжированный список профилей.

В примере баллы «Население», «Транспорт», «Экология», «Соц-об» и «Инж инф»
задаются в словаре `criteria` как параметры учебного сценария.

**Задание:** увеличьте значение «Транспорт» на один балл и сравните порядок профилей.
Какие профили изменили положение в списке и какие критерии повлияли на результат?

In [ ]:
criteria = {'Население': 4, 'Транспорт': 4, 'Экология': 3, 'Соц-об': 3, 'Инж инф': 2}
assert all(isinstance(value, int) and 0 <= value <= 5 for value in criteria.values())
ranked_profiles = evaluation.calculate_potential(criteria)
profile_table = pd.DataFrame([
    {'Профиль': name, 'Выполнено критериев': values[0], 'Взвешенное превышение': values[1]}
    for name, values in ranked_profiles
])
display(profile_table)

## 7. Дополнительные разделы
Расширим анализ с помощью данных о землепользовании, пространственном неравенстве
и инженерной инфраструктуре. Эти разделы используют внешние данные и дополнительные
слои. Ячейки выполняют расчёт при наличии соответствующих файлов.

### 7.1. Землепользование: `LandUseAssessment`
`get_landuse_data()` получает объекты OpenStreetMap в выбранном полигоне,
группирует их по типам землепользования и рассчитывает доли площади.
`plot_landuse()` отображает категории на картографической подложке.

Результат позволяет изучить состав землепользования и сопоставить его
с рассматриваемыми профилями развития территории.

Для выполнения запроса включите `RUN_OSM`. Расчёт и загрузка подложки
выполняются при подключении к интернету для выбранной территории.

In [ ]:
RUN_OSM = False  # Поставьте True, чтобы выполнить запрос и построить карту.
landuse_data = None
if RUN_OSM:
    from popframe.method.landuse_assessment import LandUseAssessment
    landuse_method = LandUseAssessment(region=region_model)
    landuse_data = landuse_method.get_landuse_data(territory.to_crs(4326))
    display(landuse_data.drop(columns='geometry'))
    landuse_method.plot_landuse(territory.to_crs(4326), landuse_data)
else:
    print('Запрос OSM выключен. Для выполнения измените RUN_OSM на True.')

### 7.2. Пространственное неравенство: `SpatialInequalityCalculator`
Дополнительный файл: **`spatial_inequality_LO_towns.geojson`** в папке `data`.
Он содержит точки населённых пунктов с рассчитанными показателями:
`Пространственное неравенство` и `<Название группы> - Неравенство`.

- `transfer_inequality_metrics_to_polygons()` рассчитывает среднее арифметическое
  показателей по точкам внутри каждого полигона агломерации.
- `get_best_territory()` выбирает территории с наименьшими значениями общего
  показателя или показателя выбранной группы. Меньшее значение соответствует меньшему неравенству.
- `get_best_group_for_territory()` определяет группу с наименьшим показателем
  в каждой территории.

Эти методы помогают сравнить территории и исследовать различия между социальными группами.

**Задание:** сравните ранжирование по общему показателю и по выбранной социальной группе.
Какие территории занимают первые места в обоих списках?

In [ ]:
from popframe.method.spatial_inequality import SpatialInequalityCalculator

inequality_path = DATA_DIR / 'spatial_inequality_LO_towns.geojson'
inequality_areas = None
if inequality_path.exists() and not base_agglomerations.empty:
    inequality_towns = gpd.read_file(inequality_path).to_crs(LOCAL_CRS)
    assert inequality_towns.geometry.geom_type.eq('Point').all(), 'Нужен слой точек.'
    metric_columns = [column for column in inequality_towns if 'неравенство' in column.lower()]
    assert metric_columns, 'В файле нет показателей неравенства.'
    assert all(pd.api.types.is_numeric_dtype(inequality_towns[column]) for column in metric_columns)
    inequality_method = SpatialInequalityCalculator(region=region_model)
    inequality_areas, inequality_stats = inequality_method.transfer_inequality_metrics_to_polygons(
        inequality_towns, base_agglomerations.reset_index(drop=True))
    display(pd.DataFrame(inequality_stats).rename(columns={
        'mean_within': 'Среднее по точкам внутри', 'mean_outside': 'Среднее по точкам вне'}))
    overall_column = 'Пространственное неравенство'
    if overall_column in inequality_areas and inequality_areas[overall_column].notna().any():
        valid_areas = inequality_areas.dropna(subset=[overall_column])
        display(inequality_method.get_best_territory(valid_areas, top_n=5).drop(columns='geometry'))
        fig, ax = plt.subplots()
        inequality_areas.plot(column=overall_column, cmap='YlOrBr', legend=True, ax=ax,
                             edgecolor='#555555', missing_kwds={'color': '#dddddd', 'hatch': '///'})
        ax.set_title('Среднее неравенство по точкам: меньше — лучше; штриховка — нет данных')
        ax.set_axis_off()
        plt.tight_layout()
        plt.show()
else:
    print('Раздел пропущен: нужен дополнительный файл и непустой слой агломераций.')

### Сравнение социальных групп
Укажите название группы без суффикса « - Неравенство»; `None` выбирает первую
доступную группу. Ранжирование выполняется по территориям со значением выбранного
показателя. Для сравнения всех групп используются строки с полным набором значений.

In [ ]:
GROUP_NAME = None
if inequality_areas is not None:
    suffix = ' - Неравенство'
    group_columns = [column for column in inequality_areas if column.endswith(suffix)
                     and column[:-len(suffix)].strip().lower() != 'итоговое']
    if group_columns:
        group_name = GROUP_NAME or group_columns[0][:-len(suffix)]
        group_column = group_name + suffix
        assert group_column in group_columns, f'Нет группы {group_name}. Доступны: {group_columns}'
        print('Выбранная группа:', group_name)
        display(inequality_method.get_best_territory(
            inequality_areas.dropna(subset=[group_column]), group_name=group_name, top_n=5
        ).drop(columns='geometry'))
        complete_areas = inequality_areas.dropna(subset=group_columns)
        if not complete_areas.empty:
            display(inequality_method.get_best_group_for_territory(complete_areas).drop(columns='geometry').head(10))
        print('Территорий с полными групповыми данными:', len(complete_areas), 'из', len(inequality_areas))
    else:
        print('В дополнительном файле нет показателей по социальным группам.')
else:
    print('Сначала нужны результаты предыдущей ячейки с дополнительными данными.')

### 7.3. Инженерная инфраструктура: `InfrastructureAnalyzer`
Дополнительный файл: **`infrastructure.geojson`**. Он содержит геометрию объектов,
категорию `type` и словарь `physical_object_type` с полем `name`.

Метод определяет, какие типы инфраструктуры представлены в окружении площадки.
Расчёт запускается при создании объекта; `get_results()` возвращает список
найденных типов и балл, равный **числу уникальных типов**.

Радиусы поиска в проекции EPSG:3857 составляют: для АЭС — 100 000,
для ГЭС — 10 000, для остальных объектов — 1 000 единиц проекции.

**Задание:** изучите найденные типы инфраструктуры и предложите профили развития
площадки, для которых такое окружение представляет интерес.

In [ ]:
infrastructure_path = DATA_DIR / 'infrastructure.geojson'
infrastructure_results = None
if infrastructure_path.exists():
    import json
    from popframe.method.engineer import InfrastructureAnalyzer
    infrastructure = gpd.read_file(infrastructure_path)
    assert {'type', 'physical_object_type'}.issubset(infrastructure.columns)
    infrastructure['physical_object_type'] = infrastructure['physical_object_type'].map(
        lambda value: json.loads(value) if isinstance(value, str) else value)
    assert infrastructure['physical_object_type'].map(lambda value: isinstance(value, dict) and 'name' in value).all()
    infrastructure_results = InfrastructureAnalyzer(infrastructure, territory).get_results()
    display(infrastructure_results.drop(columns='geometry'))
else:
    print('Раздел пропущен: дополнительный файл infrastructure.geojson не загружен.')

## 8. Сохранение результатов
Сохраняем карты-слои в GeoJSON (долгота/широта), таблицы — в CSV, а параметры,
версии и контрольную сумму учебного файла — в JSON. Это позволяет связать результат
с конкретным набором данных и условиями расчёта. Исходный pickle не изменяется.

В Colab результаты находятся в папке `/content/data/results`. Последняя ячейка
создаёт ZIP-архив; скачайте его через панель «Файлы» до завершения сессии.

### Сведения для воспроизводимости
Служебная ячейка собирает версии и источник установки. Они попадут в `parameters.json`; редактировать её не требуется.

In [ ]:
from importlib import metadata
import json

versions = {name: metadata.version(name) for name in
            ['popframe', 'geopandas', 'pandas', 'numpy', 'shapely', 'pydantic']}
direct_url = metadata.distribution('popframe').read_text('direct_url.json')
source_info = json.loads(direct_url) if direct_url else {}

### Запись файлов и создание архива
Выполните ячейку и скачайте полученный ZIP-архив.

In [ ]:
import hashlib
import shutil

RESULTS_DIR = DATA_DIR / 'results'
RESULTS_DIR.mkdir(exist_ok=True)
for filename, layer in {
    'agglomerations_base': base_agglomerations,
    'agglomerations_population': population_agglomerations,
    'agglomerations_time': time_agglomerations,
    'anchor_boundaries': anchor_boundaries,
    'territory_location': territory_scored,
}.items():
    output_path = RESULTS_DIR / f'{filename}.geojson'
    if layer.empty:
        output_path.write_text(json.dumps({'type': 'FeatureCollection', 'features': []}), encoding='utf-8')
    else:
        layer.to_crs(4326).to_file(output_path, driver='GeoJSON')

scenario_summary.to_csv(RESULTS_DIR / 'scenario_summary.csv', encoding='utf-8-sig')
status_comparison.to_csv(RESULTS_DIR / 'town_status.csv', encoding='utf-8-sig', index_label='town_id')
location_results.to_csv(RESULTS_DIR / 'territory_location.csv', encoding='utf-8-sig', index=False)
profile_table.to_csv(RESULTS_DIR / 'development_profiles.csv', encoding='utf-8-sig', index=False)
with MODEL_PATH.open('rb') as model_file:
    model_sha256 = hashlib.file_digest(model_file, 'sha256').hexdigest()
parameters = {'model_file': MODEL_NAME, 'model_sha256': model_sha256, 'versions': versions,
    'package_source': source_info, 'local_crs': str(LOCAL_CRS), 'town_id': int(selected_town_id),
    'base_time_min': BASE_TIME, 'scenario_time_min': SCENARIO_TIME,
    'scenario_population': scenario_population, 'anchor_ids': [int(value) for value in anchor_ids],
    'anchor_time_min': ANCHOR_TIME, 'criteria_assumptions': criteria,
    'site_name': territory.iloc[0]['name'], 'site_radius_m': None if hex_path.exists() else SITE_RADIUS_M}
(RESULTS_DIR / 'parameters.json').write_text(json.dumps(parameters, ensure_ascii=False, indent=2), encoding='utf-8')
archive_path = shutil.make_archive(str(DATA_DIR / 'PopFrame_results'), 'zip', RESULTS_DIR)
print('Архив результатов:', archive_path)

## 9. Самостоятельная работа

1. Выберите другой населённый пункт и обоснуйте выбор.
2. Задайте сценарий изменения его населения. Сравните категорию, границы и число
   охваченных пунктов с базой; приведите значения из таблицы результатов.
3. Измените порог времени и опишите, как изменился пространственный охват агломераций.
4. Выберите другой набор опорных пунктов и сравните полученные зоны.
5. Сформулируйте планировочный вывод на основе карт и таблиц.

**Что сдать:** архив результатов, две карты сравнения и короткий текст:
исходные данные → параметры сценария → результаты → объяснение → вывод.
Карты можно сохранить из вывода ячеек средствами браузера.

### Проверка перед сдачей

- Ячейки выполнены последовательно.
- В отчёте указаны параметры сравниваемых сценариев.
- Изменения проиллюстрированы картами и значениями из таблиц.
- Вывод связан с полученными результатами.
- Архив скачан из Colab.

### Другие возможности библиотеки
`CityPopulationScorer.run()` оценивает гексагоны по населению муниципалитетов.
Для расчёта используются муниципальные полигоны с полями `territory_id`, `population`
и гексагоны с полем `hexagon_id`.

Для подготовки модели с нуля `AdjacencyCalculator.get_dataframe()` рассчитывает
матрицу доступности по транспортному графу с весами `time_min` и полигонам
с согласованными ID. Для этого расчёта подключается `networkit`.
`PopulationFiller.fill()` распределяет население исходных территорий между пунктами
по модельным весам. В данном практикуме подготовленная модель уже загружена из pickle.

### Подсказки по работе в Colab

- Учебные файлы размещаются в папке `/content/data`.
- После перезапуска среды повторите установку, импорт и загрузку модели.
- Для обращения к файлу используйте его полное имя, включая расширение.
- Внешние запросы землепользования включаются параметром `RUN_OSM`.
- Готовый архив результатов скачивается через панель «Файлы».